# CNN_5_7ch Exp23: Soft Voting vs Soft Voting + N1 Bias (เลือกจาก Validation)
เปรียบเทียบ **Exp23 Soft Voting (เดิม)** กับ **Exp23 Soft Voting + N1 Bias ที่เลือกจาก Validation ของ Exp23 เอง** บน Validation และ Test ชุดเดียวกับที่ใช้ก่อนหน้า
(ใช้โมเดล Exp23 seed 42/43/44 ที่เทรนแล้ว ไม่ได้เทรนใหม่ จึงเป็นข้อมูล train/val/test split เดิม)

**วิธี Bias:** เฉลี่ยความน่าจะเป็นของ 3 seed → `log(p_mean)` → บวก bias เข้าคลาส N1 → `argmax`
ที่ Bias = 0 ต้องได้ผลเท่ากับ Soft Voting เดิมทุกตัวอย่าง (มี assert ตรวจ)

**การเลือก Bias:** ค้นหา grid บน Validation เท่านั้น ด้วยเกณฑ์เดียวกับ Exp17 แล้วนำค่าที่ได้ไปใช้ต่อทั้งไฟล์ ส่วน Test ใช้รายงานผลเท่านั้น ห้ามใช้ Test เลือก Bias

## 1. Import

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import gc
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.signal import resample_poly
from sklearn.metrics import (accuracy_score, cohen_kappa_score, confusion_matrix,
                             ConfusionMatrixDisplay, precision_recall_fscore_support)

## 2. ตั้งค่า

In [ ]:
PROJECT_DIR = Path("/content/drive/MyDrive/SleepStageClassificationProject")
DATA_DIR = PROJECT_DIR / "dataset" / "preprocess_microvolt_multi5"
RESULT_DIR = PROJECT_DIR / "results" / "CNN_5_7ch_Exp23_soft_N1bias_compare"
PLOT_DIR = RESULT_DIR / "per_subject_plots"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

EXP23_DIRS = {
    42: PROJECT_DIR / "results" / "CNN_5_7ch_Exp23_seed42",
    43: PROJECT_DIR / "results" / "CNN_5_7ch_Exp23_seed43",
    44: PROJECT_DIR / "results" / "CNN_5_7ch_Exp23_seed44",
}
SEEDS = [42, 43, 44]
OVERRIDE_BIAS = None             # ใส่ตัวเลขเพื่อกำหนด Bias เอง (None = เลือกอัตโนมัติจาก Validation)
BIAS_GRID = np.round(np.arange(-1.0, 1.0001, 0.05), 2)
MIN_MACRO_F1_GAIN = 0.005        # Macro-F1 ต้องดีขึ้นอย่างน้อยเท่านี้
MAX_ACC_DROP = 0.005             # Accuracy ลดได้ไม่เกินนี้
MAX_CLASS_F1_DROP = 0.02         # F1 ของ W/N2/N3/R ลดได้ไม่เกินนี้
CHECK_VAL_MODEL = False          # True = โหลดข้อมูล val แล้วให้โมเดลทำนายซ้ำเพื่อเทียบกับไฟล์เดิม (ช้ากว่า)
SHOW_PER_SUBJECT_PLOTS = True    # False = บันทึกรูปรายคนลงไฟล์อย่างเดียว ไม่แสดงใน notebook

FS_RAW, FS = 256, 100
EPOCH_LEN_RAW, EPOCH_LEN = 30 * FS_RAW, 30 * FS
SEQ_LEN = 5
SEQ_SAMPLES = SEQ_LEN * EPOCH_LEN
ALL_CHANNELS = ["F4-M1", "C4-M1", "O2-M1", "C3-M2", "E1-M2", "E2-M2", "Chin"]
SELECTED_CHANNELS = ALL_CHANNELS
CHANNEL_IDX = [ALL_CHANNELS.index(c) for c in SELECTED_CHANNELS]
N_ALL_CHANNELS, N_CHANNELS = len(ALL_CHANNELS), len(SELECTED_CHANNELS)
STAGES = ["W", "N1", "N2", "N3", "R"]
LABELS = [0, 1, 2, 3, 4]
PROB_COLS = ["prob_" + s for s in STAGES]
KEY_COLS = ["subject", "epoch_start", "epoch_end", "actual"]
BATCH_SIZE = 80
M_BASE = "Exp23_soft"

## 3. โหลด Validation จากไฟล์ที่บันทึกไว้ (val_predictions.csv)

In [ ]:
val_saved = {s: pd.read_csv(EXP23_DIRS[s] / "val_predictions.csv") for s in SEEDS}
val_base = val_saved[SEEDS[0]]
assert len(val_base) == 2878
for s, df in val_saved.items():
    assert len(df) == len(val_base)
    for col in KEY_COLS:
        assert (df[col].astype(str).to_numpy() == val_base[col].astype(str).to_numpy()).all(), f"seed {s}: {col} ไม่ตรงกัน"
    prob = df[PROB_COLS].to_numpy()
    assert np.isfinite(prob).all() and np.allclose(prob.sum(axis=1), 1, atol=1e-3)
    assert (np.array(STAGES)[prob.argmax(1)] == df["predicted"].to_numpy()).all()

y_val = val_base["actual"].map(dict(zip(STAGES, LABELS))).to_numpy().astype(int)
meta_val = val_base[["subject", "epoch_start", "epoch_end"]].copy()
P_val = {s: df[PROB_COLS].to_numpy() for s, df in val_saved.items()}
print("VAL:", len(y_val), "sequences,", meta_val["subject"].nunique(), "subjects")

## 3.1 เลือก N1 Bias จาก Validation ของ Exp23 Soft (ไม่ใช้ Test)
เกณฑ์: Macro-F1 ดีขึ้น ≥ MIN_MACRO_F1_GAIN, Accuracy ลดไม่เกิน MAX_ACC_DROP, F1 ของ W/N2/N3/R ลดไม่เกิน MAX_CLASS_F1_DROP
จากนั้นเลือกค่าที่ Macro-F1 สูงสุด ถ้าเท่ากันเลือกค่าที่ |Bias| น้อยกว่า ถ้าไม่มีค่าใดผ่านเกณฑ์ใช้ Bias = 0

In [ ]:
def soft_vote_bias(P_dict, bias=0.0):
    mean_p = np.mean([P_dict[s] for s in SEEDS], axis=0)
    scores = np.log(np.clip(mean_p, 1e-12, 1.0))
    scores[:, 1] += bias          # คอลัมน์ 1 = N1
    return scores.argmax(1)

def calc_metrics(y_true, y_pred):
    p, r, f1, sup = precision_recall_fscore_support(y_true, y_pred, labels=LABELS, zero_division=0)
    overall = {"accuracy": accuracy_score(y_true, y_pred),
               "macro_precision": p.mean(), "macro_recall": r.mean(), "macro_f1": f1.mean(),
               "cohen_kappa": cohen_kappa_score(y_true, y_pred, labels=LABELS)}
    per_class = pd.DataFrame({"precision": p, "recall": r, "f1": f1, "support": sup}, index=STAGES)
    return overall, per_class

rows = []
for b in BIAS_GRID:
    o, c = calc_metrics(y_val, soft_vote_bias(P_val, b))
    rows.append({"Bias": b, **o, **{f"{s}_F1": c.loc[s, "f1"] for s in STAGES},
                 "N1_Precision": c.loc["N1", "precision"], "N1_Recall": c.loc["N1", "recall"]})
sel_df = pd.DataFrame(rows)
base_row = sel_df[sel_df["Bias"] == 0.0].iloc[0]

ok = ((sel_df["macro_f1"] >= base_row["macro_f1"] + MIN_MACRO_F1_GAIN) &
      (sel_df["accuracy"] >= base_row["accuracy"] - MAX_ACC_DROP))
for s in ["W", "N2", "N3", "R"]:
    ok &= sel_df[f"{s}_F1"] >= base_row[f"{s}_F1"] - MAX_CLASS_F1_DROP
sel_df["eligible"] = ok
elig = sel_df[ok].copy()

if OVERRIDE_BIAS is not None:
    N1_BIAS = float(OVERRIDE_BIAS); how = "กำหนดเอง (OVERRIDE_BIAS)"
elif elig.empty:
    N1_BIAS = 0.0; how = "ไม่มีค่าใดผ่านเกณฑ์ จึงคง Bias = 0"
else:
    elig["abs_bias"] = elig["Bias"].abs()
    N1_BIAS = float(elig.sort_values(["macro_f1", "abs_bias"], ascending=[False, True]).iloc[0]["Bias"])
    how = "เลือกจาก Validation"
M_BIAS = f"Exp23_soft+N1bias{N1_BIAS:+.2f}"

print("Bias ที่ใช้:", N1_BIAS, "|", how)
print("ค่าที่ผ่านเกณฑ์ทั้งหมด:", elig["Bias"].tolist() if not elig.empty else "ไม่มี")
display(sel_df[["Bias", "accuracy", "macro_f1", "cohen_kappa", "N1_Precision", "N1_Recall", "N1_F1",
                "W_F1", "N2_F1", "N3_F1", "R_F1", "eligible"]].round(4))
sel_df.round(5).to_csv(RESULT_DIR / "bias_selection_val_grid.csv", index=False)

plt.figure(figsize=(10, 5))
for col, lab in [("macro_f1", "Macro-F1"), ("N1_F1", "N1 F1"), ("accuracy", "Accuracy")]:
    plt.plot(sel_df["Bias"], sel_df[col], marker="o", ms=3, label=lab)
plt.scatter(sel_df.loc[sel_df["eligible"], "Bias"], sel_df.loc[sel_df["eligible"], "macro_f1"],
            s=60, facecolors="none", edgecolors="green", label="ผ่านเกณฑ์")
plt.axvline(0, ls="--", alpha=0.5); plt.axvline(N1_BIAS, ls=":", color="k")
plt.xlabel("N1 Bias"); plt.ylabel("Score"); plt.grid(True, alpha=0.3); plt.legend()
plt.title(f"Exp23 Soft: การเลือก Bias บน Validation (เลือก {N1_BIAS})")
plt.tight_layout(); plt.savefig(RESULT_DIR / "bias_selection_val_plot.png", dpi=200, bbox_inches="tight"); plt.show()

## 4. เตรียมข้อมูล Test (ขั้นตอนเดียวกับไฟล์ทดสอบเดิม)

In [ ]:
def load_split(split):
    files = sorted((DATA_DIR / split).glob("*.npz"))
    if not files:
        raise FileNotFoundError(f"ไม่พบไฟล์ของ {split}: {DATA_DIR / split}")
    temp_dir = RESULT_DIR / f"temp_{split}"
    temp_dir.mkdir(parents=True, exist_ok=True)
    rows, temp_files = [], []
    for path in files:
        subject = path.stem
        subject_X, subject_rows = [], []
        with np.load(path, allow_pickle=True) as data:
            for g, stage in enumerate(data["stage"]):
                stage = str(stage)
                if stage not in STAGES or int(data["n_valid"][g]) != SEQ_LEN:
                    continue
                y = np.asarray(data["y"][g]).astype(str).ravel()
                epochs = np.asarray(data["epoch_indices"][g]).astype(int).ravel()
                x = np.asarray(data["X"][g], dtype=np.float32)
                if (len(y) != SEQ_LEN or not np.all(y == stage)
                    or len(epochs) != SEQ_LEN
                    or not np.all(np.diff(epochs) == 1)
                    or x.shape != (SEQ_LEN, EPOCH_LEN_RAW, N_ALL_CHANNELS)
                    or not np.isfinite(x).all()):
                    continue
                x = resample_poly(x[:, :, CHANNEL_IDX].reshape(-1, N_CHANNELS),
                                  FS, FS_RAW, axis=0).astype(np.float32)
                if x.shape != (SEQ_SAMPLES, N_CHANNELS):
                    continue
                subject_X.append(x)
                subject_rows.append({"subject": subject, "stage": stage,
                                     "label": STAGES.index(stage),
                                     "epoch_start": int(epochs[0]),
                                     "epoch_end": int(epochs[-1])})
        if {r["stage"] for r in subject_rows} == set(STAGES):
            temp_path = temp_dir / f"{subject}.npy"
            np.save(temp_path, np.stack(subject_X).astype(np.float32))
            temp_files.append((temp_path, len(subject_X)))
            rows.extend(subject_rows)
        del subject_X, subject_rows
    if not rows:
        raise ValueError(f"{split}: ไม่พบ Subject ที่มีครบ 5 ระยะ")
    total = sum(n for _, n in temp_files)
    X_path = RESULT_DIR / f"X_{split}.npy"
    X = np.lib.format.open_memmap(X_path, mode="w+", dtype=np.float32,
                                  shape=(total, SEQ_SAMPLES, N_CHANNELS))
    pos = 0
    for temp_path, n in temp_files:
        arr = np.load(temp_path, mmap_mode="r")
        X[pos:pos + n] = arr
        pos += n
        del arr
    X.flush()
    del X
    for temp_path, _ in temp_files:
        temp_path.unlink()
    temp_dir.rmdir()
    return np.load(X_path, mmap_mode="r"), pd.DataFrame(rows)

X_test, meta_test_full = load_split("test")
y_test = meta_test_full["label"].to_numpy(np.int32)
meta_test = meta_test_full[["subject", "epoch_start", "epoch_end"]].copy()
print("TEST:", meta_test["subject"].nunique(), "subjects,", len(meta_test), "sequences, X shape =", X_test.shape)
print(meta_test_full["stage"].value_counts().reindex(STAGES, fill_value=0))

def subject_names(split):
    return set(p.stem for p in (DATA_DIR / split).glob("*.npz"))
for other in ["train", "val"]:
    assert set(meta_test["subject"]).isdisjoint(subject_names(other)), f"test ซ้ำกับ {other}!"

if CHECK_VAL_MODEL:
    X_val, meta_val_chk = load_split("val")
    assert len(meta_val_chk) == 2878

## 5. ทำนาย Test ด้วยโมเดล Exp23 ทั้ง 3 seed

In [ ]:
def predict(model, X, chunk=1000):
    return np.concatenate([model.predict(X[i:i + chunk], batch_size=BATCH_SIZE, verbose=0)
                           for i in range(0, len(X), chunk)])

P_test = {}
for seed in SEEDS:
    model = tf.keras.models.load_model(EXP23_DIRS[seed] / "cnn5_best_model.keras", compile=False)
    assert model.input_shape[1:] == (SEQ_SAMPLES, N_CHANNELS), model.input_shape
    if CHECK_VAL_MODEL:
        pv = predict(model, X_val).argmax(1)
        old = val_saved[seed]["predicted"].map(STAGES.index).to_numpy()
        agree = (pv == old).mean()
        print(f"Exp23 seed {seed}: val ตรงกับไฟล์เดิม {agree:.4f}")
        assert agree > 0.995
    P_test[seed] = predict(model, X_test)
    t = meta_test.copy()
    t["actual"] = [STAGES[i] for i in y_test]
    t["predicted"] = [STAGES[i] for i in P_test[seed].argmax(1)]
    for i, s in enumerate(STAGES):
        t["prob_" + s] = P_test[seed][:, i]
    t.to_csv(RESULT_DIR / f"Exp23_s{seed}_test_predictions.csv", index=False)
    del model
    tf.keras.backend.clear_session(); gc.collect()
print("ทำนาย Test เสร็จ")

## 6. Soft Voting และ Soft Voting + N1 Bias

In [ ]:
DATA = {
    "VAL":  {"y": y_val,  "meta": meta_val,  "P": P_val},
    "TEST": {"y": y_test, "meta": meta_test, "P": P_test},
}
for k, d in DATA.items():
    plain = np.mean([d["P"][s] for s in SEEDS], axis=0).argmax(1)
    d["pred_base"] = soft_vote_bias(d["P"], 0.0)
    assert np.array_equal(d["pred_base"], plain), f"{k}: Bias=0 ไม่ตรงกับ Soft Voting เดิม"
    d["pred_bias"] = soft_vote_bias(d["P"], N1_BIAS)
    print(k, "Bias=0 ตรงกับ Soft Voting เดิม | คำทำนายที่เปลี่ยนเพราะ Bias:",
          int((d["pred_base"] != d["pred_bias"]).sum()), "จาก", len(d["y"]))

## 7. ฟังก์ชัน metric

In [ ]:
def compare_tables(y, pb, pn):
    ob, cb = calc_metrics(y, pb)
    on, cn = calc_metrics(y, pn)
    overall = pd.DataFrame({M_BASE: ob, M_BIAS: on}).T
    delta = (overall.loc[M_BIAS] - overall.loc[M_BASE]).to_frame("Δ (bias − base)").T
    overall = pd.concat([overall, delta])
    pc = pd.DataFrame(index=STAGES)
    for m in ["precision", "recall", "f1"]:
        pc[f"{m}_base"] = cb[m]
        pc[f"{m}_bias"] = cn[m]
        pc[f"{m}_Δ"] = cn[m] - cb[m]
    pc["support"] = cb["support"].astype(int)
    pc.loc["macro avg"] = [pc[c].mean() if c != "support" else pc["support"].sum() for c in pc.columns]
    return overall, pc

## 8. ตารางเปรียบเทียบผลรวมและรายคลาส (VAL / TEST)

In [ ]:
TABLES = {}
for k, d in DATA.items():
    ov, pc = compare_tables(d["y"], d["pred_base"], d["pred_bias"])
    TABLES[k] = (ov, pc)
    print(f"================ {k}: ภาพรวม ================")
    display(ov.round(4))
    print(f"================ {k}: รายคลาส (Precision / Recall / F1 เทียบกัน) ================")
    display(pc.round(4))
    ov.round(4).to_csv(RESULT_DIR / f"{k}_overall_comparison.csv")
    pc.round(4).to_csv(RESULT_DIR / f"{k}_per_class_comparison.csv")

print("================ สรุปรวม VAL + TEST ในตารางเดียว (Macro-F1 / Accuracy / Kappa / N1) ================")
summary = []
for k, (ov, pc) in TABLES.items():
    for m in [M_BASE, M_BIAS]:
        summary.append({"split": k, "method": m, "accuracy": ov.loc[m, "accuracy"],
                        "macro_f1": ov.loc[m, "macro_f1"], "cohen_kappa": ov.loc[m, "cohen_kappa"],
                        "N1_precision": pc.loc["N1", "precision_" + ("base" if m == M_BASE else "bias")],
                        "N1_recall": pc.loc["N1", "recall_" + ("base" if m == M_BASE else "bias")],
                        "N1_f1": pc.loc["N1", "f1_" + ("base" if m == M_BASE else "bias")]})
summary = pd.DataFrame(summary)
display(summary.round(4))
summary.round(4).to_csv(RESULT_DIR / "summary_val_test.csv", index=False)

## 9. Confusion Matrix (จำนวน / normalize / ผลต่าง)

In [ ]:
for k, d in DATA.items():
    cm_b = confusion_matrix(d["y"], d["pred_base"], labels=LABELS)
    cm_n = confusion_matrix(d["y"], d["pred_bias"], labels=LABELS)
    cmn_b = confusion_matrix(d["y"], d["pred_base"], labels=LABELS, normalize="true")
    cmn_n = confusion_matrix(d["y"], d["pred_bias"], labels=LABELS, normalize="true")
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    ConfusionMatrixDisplay(cm_b, display_labels=STAGES).plot(ax=axes[0, 0], cmap="Blues", colorbar=False)
    axes[0, 0].set_title(f"{M_BASE} (count)")
    ConfusionMatrixDisplay(cm_n, display_labels=STAGES).plot(ax=axes[0, 1], cmap="Blues", colorbar=False)
    axes[0, 1].set_title(f"{M_BIAS} (count)")
    ConfusionMatrixDisplay(cmn_b, display_labels=STAGES).plot(ax=axes[1, 0], cmap="Blues", values_format=".2f", colorbar=False)
    axes[1, 0].set_title(f"{M_BASE} (normalized by actual)")
    ConfusionMatrixDisplay(cmn_n, display_labels=STAGES).plot(ax=axes[1, 1], cmap="Blues", values_format=".2f", colorbar=False)
    axes[1, 1].set_title(f"{M_BIAS} (normalized by actual)")
    plt.suptitle(f"{k}: Confusion Matrix"); plt.tight_layout()
    plt.savefig(RESULT_DIR / f"{k}_confusion_matrices.png", dpi=200, bbox_inches="tight"); plt.show()

    diff = pd.DataFrame(cm_n - cm_b, index=[f"จริง {s}" for s in STAGES], columns=[f"ทาย {s}" for s in STAGES])
    print(f"{k}: ผลต่างจำนวน (bias − base) แถว=จริง คอลัมน์=ทาย")
    display(diff)
    pd.DataFrame(cm_b, index=STAGES, columns=STAGES).to_csv(RESULT_DIR / f"{k}_cm_base.csv")
    pd.DataFrame(cm_n, index=STAGES, columns=STAGES).to_csv(RESULT_DIR / f"{k}_cm_bias.csv")
    diff.to_csv(RESULT_DIR / f"{k}_cm_diff.csv")

## 10. กราฟเปรียบเทียบรายคลาส และเส้นโค้งตามค่า Bias

In [ ]:
x = np.arange(len(STAGES)); w = 0.38
for k, (ov, pc) in TABLES.items():
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    for ax, m, name in zip(axes, ["precision", "recall", "f1"], ["Precision", "Recall", "F1"]):
        ax.bar(x - w/2, pc.loc[STAGES, f"{m}_base"], w, label=M_BASE)
        ax.bar(x + w/2, pc.loc[STAGES, f"{m}_bias"], w, label=M_BIAS)
        ax.set_xticks(x); ax.set_xticklabels(STAGES); ax.set_ylim(0, 1)
        ax.set_title(f"{k}: {name} per class"); ax.grid(True, axis="y", alpha=0.3)
    axes[0].legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(RESULT_DIR / f"{k}_per_class_bars.png", dpi=200, bbox_inches="tight"); plt.show()

bias_grid = BIAS_GRID
rows = []
for k, d in DATA.items():
    for b in bias_grid:
        o, c = calc_metrics(d["y"], soft_vote_bias(d["P"], b))
        rows.append({"split": k, "bias": b, **o, "N1_f1": c.loc["N1", "f1"]})
grid_df = pd.DataFrame(rows)
grid_df.to_csv(RESULT_DIR / "bias_curve_val_test.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), sharey=True)
for ax, k in zip(axes, ["VAL", "TEST"]):
    g = grid_df[grid_df["split"] == k]
    for col, lab in [("macro_f1", "Macro-F1"), ("N1_f1", "N1 F1"), ("accuracy", "Accuracy"), ("cohen_kappa", "Kappa")]:
        ax.plot(g["bias"], g[col], marker="o", ms=3, label=lab)
    ax.axvline(0, ls="--", alpha=0.5); ax.axvline(N1_BIAS, ls=":", color="k", alpha=0.8)
    ax.set_title(f"{k}: ผลตามค่า N1 Bias (เส้นประจุด = Bias ที่เลือก {N1_BIAS})"); ax.set_xlabel("N1 Bias"); ax.grid(True, alpha=0.3)
axes[0].legend(); axes[0].set_ylabel("Score")
plt.tight_layout(); plt.savefig(RESULT_DIR / "bias_curve_val_test.png", dpi=200, bbox_inches="tight"); plt.show()

## 11. Bias เปลี่ยนคำทำนายอย่างไร (ได้/เสียกี่ตัวอย่าง)

In [ ]:
for k, d in DATA.items():
    y, pb, pn = d["y"], d["pred_base"], d["pred_bias"]
    changed = pb != pn
    gained = changed & (pn == y)           # เดิมผิด -> ถูกหลังบวก bias
    lost = changed & (pb == y)             # เดิมถูก -> ผิดหลังบวก bias
    still_wrong = changed & (pb != y) & (pn != y)
    print(f"{k}: เปลี่ยน {changed.sum()} | ถูกเพิ่ม +{gained.sum()} | ถูกลดลง −{lost.sum()} | ผิดเหมือนเดิม {still_wrong.sum()} | สุทธิ {gained.sum() - lost.sum():+d}")
    flip = pd.crosstab(pd.Series(np.array(STAGES)[pb[changed]], name="เดิมทาย"),
                       pd.Series(np.array(STAGES)[y[changed]], name="คลาสจริง"))
    print("ตัวอย่างที่เปลี่ยน (ทุกตัวเปลี่ยนไปเป็น N1): แถว=คำทำนายเดิม คอลัมน์=คลาสจริง")
    display(flip)

## 12. ผลรายคน (Per-subject) ของไฟล์ที่ทดสอบ

In [ ]:
def per_subject_table(split):
    d = DATA[split]
    rows = []
    for subj, idx in d["meta"].groupby("subject").indices.items():
        y = d["y"][idx]
        row = {"subject": subj, "n_seq": len(idx)}
        for tag, key in [("base", "pred_base"), ("bias", "pred_bias")]:
            p = d[key][idx]
            o, c = calc_metrics(y, p)
            row[f"acc_{tag}"] = o["accuracy"]
            row[f"macro_f1_{tag}"] = o["macro_f1"]
            row[f"kappa_{tag}"] = o["cohen_kappa"]
            row[f"N1_f1_{tag}"] = c.loc["N1", "f1"]
            row[f"N1_recall_{tag}"] = c.loc["N1", "recall"]
        for m in ["acc", "macro_f1", "kappa", "N1_f1", "N1_recall"]:
            row[f"{m}_Δ"] = row[f"{m}_bias"] - row[f"{m}_base"]
        rows.append(row)
    return pd.DataFrame(rows).sort_values("subject").reset_index(drop=True)

SUBJ = {}
for k in DATA:
    SUBJ[k] = per_subject_table(k)
    print(f"================ {k}: ผลรายคน ================")
    display(SUBJ[k].round(4))
    SUBJ[k].round(4).to_csv(RESULT_DIR / f"{k}_per_subject_comparison.csv", index=False)
    t = SUBJ[k]
    print(f"{k}: Macro-F1 ดีขึ้น {int((t['macro_f1_Δ'] > 0).sum())} คน | แย่ลง {int((t['macro_f1_Δ'] < 0).sum())} คน | เท่าเดิม {int((t['macro_f1_Δ'] == 0).sum())} คน (จาก {len(t)} คน)")
    print(f"{k}: Δ Macro-F1 เฉลี่ยต่อคน = {t['macro_f1_Δ'].mean():+.4f} | Δ N1 F1 เฉลี่ยต่อคน = {t['N1_f1_Δ'].mean():+.4f}")

In [ ]:
for k, t in SUBJ.items():
    n = len(t); xs = np.arange(n)
    fig, axes = plt.subplots(3, 1, figsize=(max(10, n * 0.5), 12), sharex=True)
    for ax, m, name in zip(axes[:2], ["macro_f1", "N1_f1"], ["Macro-F1", "N1 F1"]):
        ax.bar(xs - 0.2, t[f"{m}_base"], 0.4, label=M_BASE)
        ax.bar(xs + 0.2, t[f"{m}_bias"], 0.4, label=M_BIAS)
        ax.set_ylabel(name); ax.set_ylim(0, 1); ax.grid(True, axis="y", alpha=0.3)
        ax.set_title(f"{k}: {name} รายคน")
    axes[0].legend(fontsize=8)
    colors = np.where(t["macro_f1_Δ"] >= 0, "tab:green", "tab:red")
    axes[2].bar(xs, t["macro_f1_Δ"], color=colors)
    axes[2].axhline(0, color="k", lw=0.8); axes[2].set_ylabel("Δ Macro-F1 (bias − base)")
    axes[2].set_title(f"{k}: Δ Macro-F1 รายคน (เขียว=ดีขึ้น แดง=แย่ลง)"); axes[2].grid(True, axis="y", alpha=0.3)
    axes[2].set_xticks(xs); axes[2].set_xticklabels(t["subject"], rotation=75, ha="right", fontsize=8)
    plt.tight_layout()
    plt.savefig(RESULT_DIR / f"{k}_per_subject_bars.png", dpi=200, bbox_inches="tight"); plt.show()

### กราฟ Stage ของแต่ละคน (Actual / Soft / Soft+N1 Bias)
ข้อมูลเป็นหน้าต่าง 5 epoch ที่สุ่มตามระยะ ไม่ได้ต่อเนื่องทั้งคืน จึงพล็อตเป็นจุดตามตำแหน่ง `epoch_start` (จุดแดง = ทำนายผิด)

In [ ]:
STAGE_Y = {i: i for i in range(5)}

def plot_subject(split, subj):
    d = DATA[split]
    idx = np.where(d["meta"]["subject"].to_numpy() == subj)[0]
    idx = idx[np.argsort(d["meta"]["epoch_start"].to_numpy()[idx])]
    ep = d["meta"]["epoch_start"].to_numpy()[idx]
    y, pb, pn = d["y"][idx], d["pred_base"][idx], d["pred_bias"][idx]
    fig, axes = plt.subplots(3, 1, figsize=(13, 6.5), sharex=True)
    for ax, series, title in zip(axes, [y, pb, pn], ["Actual", M_BASE, M_BIAS]):
        ax.plot(ep, series, color="lightgray", lw=0.8, zorder=1)
        ok = (series == y)
        ax.scatter(ep[ok], series[ok], s=14, color="tab:blue", zorder=2)
        if title != "Actual":
            ax.scatter(ep[~ok], series[~ok], s=18, color="tab:red", zorder=3)
        ax.set_yticks(LABELS); ax.set_yticklabels(STAGES); ax.invert_yaxis(); ax.grid(True, alpha=0.3)
        ax.set_ylabel(title, fontsize=8)
    ob, _ = calc_metrics(y, pb); on, _ = calc_metrics(y, pn)
    axes[0].set_title(f"{split} | {subj} | acc {ob['accuracy']:.3f}→{on['accuracy']:.3f} | macro-F1 {ob['macro_f1']:.3f}→{on['macro_f1']:.3f}")
    axes[-1].set_xlabel("epoch_start")
    plt.tight_layout()
    plt.savefig(PLOT_DIR / f"{split}_{subj}_stage_plot.png", dpi=150, bbox_inches="tight")
    if SHOW_PER_SUBJECT_PLOTS:
        plt.show()
    else:
        plt.close(fig)

for split in DATA:
    for subj in sorted(DATA[split]["meta"]["subject"].unique()):
        plot_subject(split, subj)
print("บันทึกกราฟรายคนที่:", PLOT_DIR)

## 13. บันทึกคำทำนายทั้งหมด

In [ ]:
for k, d in DATA.items():
    t = d["meta"].copy()
    t["actual"] = np.array(STAGES)[d["y"]]
    t["pred_soft"] = np.array(STAGES)[d["pred_base"]]
    t["pred_soft_N1bias"] = np.array(STAGES)[d["pred_bias"]]
    t["changed"] = d["pred_base"] != d["pred_bias"]
    t.to_csv(RESULT_DIR / f"{k}_ensemble_predictions.csv", index=False)
print("บันทึกผลทั้งหมดที่:", RESULT_DIR)